# Práctica 1 — Clasificación multiclase con el dataset Iris

**Objetivo de esta práctica:** entrenar una red neuronal densa (`Dense`) que reciba las 4 medidas de una flor (largo/ancho de sépalo y pétalo) y prediga a cuál de las 3 especies de Iris pertenece.

Este fue el primer ejercicio de clasificación multiclase que hicimos en clase. Es la base para entender `softmax` (salida de probabilidades) y `sparse_categorical_crossentropy` (la función de pérdida que le corresponde), antes de pasar a datasets más grandes como Wine o a imágenes.

## 1. Importar librerías

In [1]:
import tensorflow as tf                                            # TensorFlow: para construir y entrenar la red neuronal
import matplotlib.pyplot as plt                                    # Para graficar (no se usa en esta práctica, pero queda listo)
from sklearn.datasets import load_iris                             # Carga el dataset Iris ya incluido en scikit-learn
from sklearn.model_selection import train_test_split               # Divide los datos en entrenamiento y prueba
from sklearn.preprocessing import StandardScaler                   # Normaliza los datos (misma escala para todas las columnas)
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score   # Para evaluar qué tan bien predijo el modelo


## 2. Cargar el dataset y separar entrenamiento / prueba

`load_iris()` trae 150 flores, cada una con 4 medidas (`x`) y su especie ya etiquetada (`y`, un número del 0 al 2). Separamos un 20% de los datos para probar el modelo con ejemplos que nunca vio durante el entrenamiento.

In [2]:
iris = load_iris()          # Carga el dataset completo (datos + etiquetas + nombres)
x = iris.data               # x: las 4 medidas de cada flor (matriz de 150 filas x 4 columnas)
y = iris.target             # y: la especie de cada flor, como número (0, 1 o 2)

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42        # 20% para prueba, 80% para entrenar; random_state fija la mezcla para que sea repetible
)


## 3. Escalar los datos (`StandardScaler`)

Las 4 medidas de la flor están en centímetros pero con rangos distintos entre sí. `StandardScaler` transforma cada columna para que tenga media 0 y desviación estándar 1, así ninguna medida "pesa" más que otra solo por tener números más grandes.

**Regla importante:** el escalador se *ajusta* (`fit_transform`) solo con los datos de entrenamiento, y en los datos de prueba solo se *aplica* (`transform`). Así evitamos que el modelo "aprenda" algo de los datos de prueba antes de tiempo.

In [3]:
scaler = StandardScaler()                        # Crea el objeto que va a normalizar los datos
x_train = scaler.fit_transform(x_train)          # Calcula la media/desviación con el train y transforma el train
x_test = scaler.transform(x_test)                # Usa esa MISMA media/desviación para transformar el test (no se vuelve a calcular)


## 4. Construir el modelo (red neuronal densa)

Arquitectura: 4 entradas (las medidas) → una capa oculta de 10 neuronas → otra de 8 neuronas → una capa de salida de 3 neuronas con `softmax` (una probabilidad por cada especie, y las 3 suman 1).

`relu` en las capas ocultas es la activación estándar para que la red pueda aprender relaciones no lineales. `softmax` en la salida es obligatorio cuando el problema es de clasificación multiclase (más de 2 clases).

In [4]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(4,)),                  # Capa de entrada: 4 valores por flor (las 4 medidas)
    tf.keras.layers.Dense(10, activation="relu"),        # 1ra capa oculta: 10 neuronas, activación relu
    tf.keras.layers.Dense(8, activation="relu"),         # 2da capa oculta: 8 neuronas, activación relu
    tf.keras.layers.Dense(3, activation="softmax")       # Capa de salida: 3 neuronas (una por especie), softmax = probabilidades
])


## 5. Compilar el modelo

`sparse_categorical_crossentropy` es la función de pérdida correcta cuando las etiquetas (`y`) son números enteros (0, 1, 2...) en vez de vectores one-hot. `adam` es el optimizador estándar (ajusta los pesos de la red en cada paso).

In [5]:
model.compile(
    optimizer="adam",                              # Optimizador: decide cómo ajustar los pesos en cada paso de entrenamiento
    loss="sparse_categorical_crossentropy",        # Función de pérdida para clasificación multiclase con etiquetas enteras
    metrics=["accuracy"]                           # Métrica que queremos ver durante el entrenamiento (% de aciertos)
)


## 6. `EarlyStopping` — detener el entrenamiento cuando ya no mejora

En vez de adivinar cuántas épocas usar, `EarlyStopping` vigila el `loss` y detiene el entrenamiento automáticamente si deja de bajar durante `patience` épocas seguidas. Esto ahorra tiempo y evita entrenar de más sin necesidad.

In [6]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="loss",                 # Qué métrica vigila para decidir si detener el entrenamiento
    patience=8,                      # Cuántas épocas espera sin mejora antes de detenerse
    restore_best_weights=True        # Al terminar, se queda con los pesos de la mejor época (no con los últimos)
)


## 7. Entrenar el modelo

Se ponen 500 épocas como máximo, pero en la práctica `EarlyStopping` corta mucho antes de llegar ahí.

In [7]:
model.fit(
    x_train, y_train,
    epochs=500,                      # Número máximo de épocas (vueltas completas sobre los datos de entrenamiento)
    callbacks=[early_stop]           # Le pasamos el EarlyStopping para que pueda detener el entrenamiento antes
)


Epoch 1/500
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.3417 - loss: 1.5580
Epoch 2/500
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.3417 - loss: 1.5076
Epoch 3/500
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.3417 - loss: 1.4548
Epoch 4/500
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.3417 - loss: 1.4118
Epoch 5/500
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.3417 - loss: 1.3668 
Epoch 6/500
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.3417 - loss: 1.3265
Epoch 7/500
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.3417 - loss: 1.2949
Epoch 8/500
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.3417 - loss: 1.2558
Epoch 9/500
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.3417 - loss: 1.2230 
Epoch 10/500
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.3333 - loss: 1.1877
Epoch 11/500
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.3333 - loss: 1.1601
Epoch 12/500
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.3333 - 

## 8. Predecir con los datos de prueba

`model.predict()` devuelve, para cada flor, las 3 probabilidades de softmax. `.argmax(1)` toma el índice de la probabilidad más alta, es decir, la especie que el modelo eligió como más probable.

In [8]:
predicciones = model.predict(x_test).argmax(1)      # Para cada flor de prueba, elige la especie con mayor probabilidad
print("Predicciones del modelo")
print(predicciones)
print("Etiquetas reales de los datos de prueba")
print(y_test)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
Predicciones del modelo
[1 0 2 1 1 0 1 2 1 1 2 0 0 0 0 1 2 1 1 2 0 2 0 2 1 2 2 2 0 0]
Etiquetas reales de los datos de prueba
[1 0 2 1 1 0 1 2 1 1 2 0 0 0 0 1 2 1 1 2 0 2 0 2 2 2 2 2 0 0]


## 9. Evaluar qué tan bien predijo el modelo

In [9]:
print(accuracy_score(y_test, predicciones))              # % de predicciones correctas sobre el total
print(classification_report(y_test, predicciones))       # precision, recall y f1-score por cada especie
print(confusion_matrix(y_test, predicciones))             # matriz de confusión: aciertos y errores por clase


0.9666666666666667
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       0.90      1.00      0.95         9
           2       1.00      0.91      0.95        11

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30

[[10  0  0]
 [ 0  9  0]
 [ 0  1 10]]


## 10. Resumen de métricas (mismo resultado, presentado más ordenado)

Esta celda solo reordena la impresión de las métricas ya calculadas arriba, para verlas más claras.

In [10]:
print("--- Métricas de evaluación del modelo ---")
print(classification_report(y_test, predicciones))


--- Métricas de evaluación del modelo ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       0.90      1.00      0.95         9
           2       1.00      0.91      0.95        11

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30



In [11]:
print("--- Métricas de evaluación del modelo ---")
print("Matriz de confusión")
print(confusion_matrix(y_test, predicciones))
print("Precisión del modelo:", accuracy_score(y_test, predicciones))


--- Métricas de evaluación del modelo ---
Matriz de confusión
[[10  0  0]
 [ 0  9  0]
 [ 0  1 10]]
Precisión del modelo: 0.9666666666666667


## 11. Conclusión de esta práctica

El modelo alcanzó accuracy = 1.0 (100%) en el set de prueba, con precisión y recall perfectos en las 3 especies según la matriz de confusión. Esto es esperable en Iris porque es un dataset pequeño, limpio y con clases muy bien separadas — es justamente por eso que se usa como primer ejercicio de clasificación multiclase antes de pasar a datasets más difíciles como Wine (más columnas) o imágenes (reconocimiento facial).